In [1]:
import torch
import math
import torch.nn as nn

In [6]:
#causal lm decoder Block

class SimpleDecoderLayer(nn.Module):
    def __init__(self,hidden_dim,head_num,attention_dropout_rate=0.1):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.head_num = head_num
        self.attention_dropout_rate = attention_dropout_rate
        
        self.head_dim = hidden_dim//head_num
        
        #layer = mha + ffn
        self.q_proj = nn.Linear(hidden_dim,hidden_dim)
        self.k_proj = nn.Linear(hidden_dim,hidden_dim)
        self.v_proj = nn.Linear(hidden_dim,hidden_dim)
        self.o_proj = nn.Linear(hidden_dim,hidden_dim)
        self.dropout = nn.Dropout(attention_dropout_rate)
        self.att_ln = nn.LayerNorm(hidden_dim,eps=0.0000001) #eps 防止除0
        
        #ffn (升维 -》 降维 -》ln)
        self.up_proj = nn.Linear(hidden_dim, hidden_dim * 4)
        self.down_proj = nn.Linear(hidden_dim * 4,hidden_dim)
        
        self.act_fn = nn.GELU()
        self.drop_ffn = nn.Dropout(0.1)
        self.ffn_ln = nn.LayerNorm(hidden_dim,eps = 0.0000001)
        
    def mha(self,x,mask = None):
        # (b,s,h) -> (b,head_num,s,head_dim)
        batch , seq , _  = x.size()
        self.key = self.k_proj(x)
        self.query = self.q_proj(x)
        self.value = self.v_proj(x)
        
        mid_key = self.key.view(batch,seq,self.head_num,-1).transpose(1,2) #(b , nums_head, s , head_dim)
        mid_query = self.query.view(batch,seq,self.head_num,-1).transpose(1,2)
        mid_value = self.value.view(batch,seq,self.head_num,-1).transpose(1,2)
        
        weight = mid_query @ mid_key.transpose(2,3) / math.sqrt(self.head_dim) #( b, nums_head, s, s)
        
        if mask is None:
            attention_mask = torch.ones_like(weight).tril()
            weight = weight.masked_fill(
                attention_mask==0,float('-inf')
            )
        else:
            attention_mask = mask.tril()
            weight = weight.masked_fill(
                attention_mask==0,float('-inf')
            )
        attention_weight = torch.softmax(weight,dim=-1)
        attention_weight = self.dropout(attention_weight)
        
        mid_out = attention_weight @ mid_value #(b , nums_head, s , head_dim)
        mid_out = mid_out.transpose(1,2).contiguous()
        mid_out = mid_out.view(batch,seq,-1)
        
        output = self.o_proj(mid_out)
        
        return output
        
    
    def ffn(self,x):
        up = self.up_proj(x)
        up = self.act_fn(up)
        down = self.down_proj(up)
        down = self.drop_ffn(down)
        # post layer_norm
        return self.ffn_ln(x + down)
       
       
    def forward(self,x,attention_mask = None):
        x = self.mha(x,attention_mask)
        x = self.ffn(x)
        return x
        
        

class Decoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer_list = nn.ModuleList(
            [
                SimpleDecoderLayer(64,8) for i in range(5)
            ]
        )
        self.input = nn.Embedding(12,64)
        self.output = nn.Linear(64,12)
        
    def forward(self,x,mask=None):
        #(b,s)
        x = self.input(x)
        for i,l in enumerate(self.layer_list):
            x = l(x,mask)
        print(x.shape)
        output = self.output(x)
        return torch.softmax(output,dim=-1)

        

In [7]:
x = torch.rand(3, 4, 64)
net = SimpleDecoderLayer(64, 8)
mask = (
    torch.tensor([[1, 1, 1, 1], [1, 1, 0, 0], [1, 1, 1, 0]])
    .unsqueeze(1)
    .unsqueeze(2)
    .repeat(1, 8, 4, 1)
)

net(x, mask).shape

torch.Size([3, 4, 64])